# Deep dive 6/13: `daily_exchange_rates`

First REFERENCE-kind table (not a dimension or fact) -- and the first table built with a hand-written function (`build_fx_rates_table` in `silver.py`) instead of a `table_specs.py` `ColumnSpec` list. Two things make it different from every table so far:

1. **The Silver table is renamed and has renamed columns.** `bronze.daily_exchange_rates` -> `silver.dim_fx_rates`; `date` -> `rate_date`, `source` -> `rate_source`. Every other column keeps its name.
2. **The primary key is composite**: `(date, source_currency, target_currency)`, not a single ID column -- `contracts.py` already reflects this via `key_fields=("date", "source_currency", "target_currency")`.
3. **A derived view sits on top of it**: `silver.v_fx_latest_to_usd` (latest USD rate per source currency), used by `products`' currency-conversion logic since a snapshot dimension has no natural transaction date to join on. Section 9 checks the view too, since a bug there would silently corrupt every USD-converted `products`/other-table amount downstream.

Dictionary declares 3,000 rows, Source: Reference, Partition: daily. No FK relationships point at this table (it's joined by `date`/currency, not by ID) -- confirmed in the FK Relationships section of the dictionary, `daily_exchange_rates` doesn't appear as a parent anywhere.

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. A look at the table -- sample rows, Bronze and Silver side by side

In [ ]:
print("Bronze sample:")
display(q_full("SELECT * FROM bronze.daily_exchange_rates ORDER BY date, source_currency, target_currency LIMIT 20"))

In [ ]:
print("Silver sample (note the renamed columns: date -> rate_date, source -> rate_source):")
display(q_full("SELECT * FROM silver.dim_fx_rates ORDER BY rate_date, source_currency, target_currency LIMIT 20"))

## 1. Column inventory -- names AND types, Bronze vs. Silver (accounting for the rename)

In [ ]:
# silver.py's rename map: bronze.date -> silver.rate_date, bronze.source -> silver.rate_source.
# Everything else keeps its name.
expected_silver_columns = [
    "rate_date", "source_currency", "target_currency", "exchange_rate", "buy_rate", "sell_rate", "rate_source",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_fx_rates'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_fx_rates:", missing or "none")
print("Extra in silver.dim_fx_rates:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='daily_exchange_rates' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_fx_rates' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation + composite-key dedup

The `QUALIFY row_number() OVER (PARTITION BY date, source_currency, target_currency ORDER BY _ingested_at DESC) = 1` in `build_fx_rates_table` should collapse any duplicate rows sharing the same composite key, keeping the most recently ingested one.

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.daily_exchange_rates")["n"][0]
bronze_distinct_key = q("""
    SELECT count(*) AS n FROM (
        SELECT DISTINCT TRY_CAST(date AS DATE), trim(source_currency), trim(target_currency)
        FROM bronze.daily_exchange_rates
    )
""")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_fx_rates")["n"][0]
silver_distinct_key = q("""
    SELECT count(*) AS n FROM (
        SELECT DISTINCT rate_date, source_currency, target_currency FROM silver.dim_fx_rates
    )
""")["n"][0]

print(f"bronze.daily_exchange_rates: {bronze_total:,} rows, {bronze_distinct_key:,} distinct (date, source_currency, target_currency)")
print(f"silver.dim_fx_rates:         {silver_total:,} rows, {silver_distinct_key:,} distinct (rate_date, source_currency, target_currency)")
print()
print("Dictionary declares 3,000 rows -- matches" if bronze_total == 3000 else f"Dictionary declares 3,000 rows -- MISMATCH, Bronze has {bronze_total}")
print("Duplicate raw rows in Bronze (same composite key, multiple rows):", bronze_total - bronze_distinct_key)
print("OK -- Silver row count matches Bronze distinct composite-key count" if silver_total == bronze_distinct_key
      else "MISMATCH -- investigate: dedup may not be working as intended")

## 2.1 A closer look at any duplicated composite keys

If section 2 found duplicates, confirm Silver actually kept the most-recently-ingested row for each one (not an arbitrary one), same verification pattern used for `customers`/`products` dedup.

In [ ]:
dupe_keys = q("""
    SELECT TRY_CAST(date AS DATE) AS date, trim(source_currency) AS source_currency,
           trim(target_currency) AS target_currency, count(*) AS n
    FROM bronze.daily_exchange_rates
    GROUP BY 1, 2, 3
    HAVING count(*) > 1
    ORDER BY n DESC
""")
print(f"{len(dupe_keys)} composite keys with more than one raw Bronze row")
display(dupe_keys.head(20))

In [ ]:
# For one duplicated key (if any), confirm Silver kept the max(_ingested_at) row's values.
if len(dupe_keys) > 0:
    d, sc, tc = dupe_keys.iloc[0][["date", "source_currency", "target_currency"]]
    print(f"Bronze rows for ({d}, {sc}, {tc}):")
    display(q_full(f"""
        SELECT *, row_number() OVER (ORDER BY _ingested_at DESC) AS rn
        FROM bronze.daily_exchange_rates
        WHERE TRY_CAST(date AS DATE) = '{d}' AND trim(source_currency) = '{sc}' AND trim(target_currency) = '{tc}'
        ORDER BY _ingested_at DESC
    """))
    print("Silver's row for the same key:")
    display(q_full(f"""
        SELECT * FROM silver.dim_fx_rates
        WHERE rate_date = '{d}' AND source_currency = '{sc}' AND target_currency = '{tc}'
    """))
else:
    print("No duplicated composite keys -- nothing to verify.")

## 3. NOT NULL audit -- Bronze raw vs. Silver typed

Dictionary NOT NULL: `date`, `source_currency`, `target_currency`, `exchange_rate` (the composite PK plus the rate itself). Nullable: `buy_rate`, `sell_rate`, `source`.

In [ ]:
declared_not_null_bronze = ["date", "source_currency", "target_currency", "exchange_rate"]
declared_not_null_silver = ["rate_date", "source_currency", "target_currency", "exchange_rate"]

rows = []
for bcol, scol in zip(declared_not_null_bronze, declared_not_null_silver):
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.daily_exchange_rates
        WHERE {bcol} IS NULL OR trim(CAST({bcol} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.dim_fx_rates WHERE {scol} IS NULL
    """)["n"][0]
    rows.append({"bronze_column": bcol, "silver_column": scol, "declared": "NOT NULL",
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

## 4. Nullable columns (`buy_rate`/`sell_rate`/`source`->`rate_source`) -- population rate, Bronze vs. Silver

In [ ]:
nullable_pairs = [("buy_rate", "buy_rate"), ("sell_rate", "sell_rate"), ("source", "rate_source")]
rows = []
total = q("SELECT count(*) AS n FROM bronze.daily_exchange_rates")["n"][0]
for bcol, scol in nullable_pairs:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.daily_exchange_rates
        WHERE {bcol} IS NOT NULL AND trim(CAST({bcol} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.dim_fx_rates WHERE {scol} IS NOT NULL
    """)["n"][0]
    rows.append({"bronze_column": bcol, "silver_column": scol, "total": total,
                 "bronze_populated": bronze_populated, "silver_populated": silver_populated,
                 "cast_dropped_values": bronze_populated - silver_populated})

pd.DataFrame(rows)

## 5. `source_currency`/`target_currency` -- domain check

Dictionary declares 4 currencies for the whole dataset: MXN, COP, ARS, USD. Not currently gated in `contracts.py`.

In [ ]:
print("source_currency (Bronze):")
display(q("SELECT source_currency, count(*) AS n FROM bronze.daily_exchange_rates GROUP BY 1 ORDER BY n DESC"))
print("target_currency (Bronze):")
display(q("SELECT target_currency, count(*) AS n FROM bronze.daily_exchange_rates GROUP BY 1 ORDER BY n DESC"))
print("Distinct (source_currency, target_currency) pairs and their row counts:")
display(q("SELECT source_currency, target_currency, count(*) AS n FROM bronze.daily_exchange_rates GROUP BY 1, 2 ORDER BY n DESC"))

In [ ]:
declared_currencies = {"MXN", "COP", "ARS", "USD"}
real_source = set(q("SELECT DISTINCT source_currency AS v FROM bronze.daily_exchange_rates")["v"])
real_target = set(q("SELECT DISTINCT target_currency AS v FROM bronze.daily_exchange_rates")["v"])
print("source_currency values NOT in the declared 4-currency set:", real_source - declared_currencies or "none -- full match")
print("target_currency values NOT in the declared 4-currency set:", real_target - declared_currencies or "none -- full match")
print()
print("Any row where source_currency == target_currency (a same-to-same 'rate')?")
display(q("SELECT * FROM bronze.daily_exchange_rates WHERE source_currency = target_currency LIMIT 10"))

## 6. `exchange_rate`/`buy_rate`/`sell_rate` -- cast correctness, range sanity, internal consistency

In [ ]:
q("""
    SELECT
        typeof(exchange_rate) AS rate_type, typeof(buy_rate) AS buy_type, typeof(sell_rate) AS sell_type,
        min(exchange_rate) AS min_rate, max(exchange_rate) AS max_rate,
        count(*) FILTER (WHERE exchange_rate <= 0) AS non_positive_rates,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND buy_rate <= 0) AS non_positive_buy,
        count(*) FILTER (WHERE sell_rate IS NOT NULL AND sell_rate <= 0) AS non_positive_sell
    FROM silver.dim_fx_rates
    GROUP BY ALL
""")

In [ ]:
# By currency pair: does the magnitude look like a real-world rate? (rough sanity bounds, not a hard check)
q("""
    SELECT source_currency, target_currency,
           count(*) AS n, round(min(exchange_rate), 4) AS min_rate, round(max(exchange_rate), 4) AS max_rate,
           round(avg(exchange_rate), 4) AS avg_rate
    FROM silver.dim_fx_rates
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

In [ ]:
# buy_rate <= exchange_rate <= sell_rate is the normal convention (bank buys low, sells high);
# check whether that holds, and how often buy_rate/sell_rate are populated together vs. exchange_rate alone.
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND sell_rate IS NOT NULL) AS both_populated,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND sell_rate IS NOT NULL AND buy_rate > sell_rate) AS buy_gt_sell,
        count(*) FILTER (WHERE buy_rate IS NOT NULL AND buy_rate > exchange_rate) AS buy_gt_mid,
        count(*) FILTER (WHERE sell_rate IS NOT NULL AND sell_rate < exchange_rate) AS sell_lt_mid
    FROM silver.dim_fx_rates
""")

## 7. `date`/`rate_date` -- cast correctness, coverage, window check

In [ ]:
q("""
    SELECT
        typeof(rate_date) AS silver_type,
        count(*) FILTER (WHERE rate_date IS NULL) AS nulls,
        min(rate_date) AS earliest, max(rate_date) AS latest,
        count(DISTINCT rate_date) AS distinct_dates
    FROM silver.dim_fx_rates
    GROUP BY ALL
""")

In [ ]:
# Same window check as every other table.
q("""
    SELECT
        count(*) FILTER (WHERE rate_date < DATE '2023-06-17') AS before_window,
        count(*) FILTER (WHERE rate_date > DATE '2026-06-17') AS after_window
    FROM silver.dim_fx_rates
""")

In [ ]:
# Coverage check: for each currency pair, is there roughly one rate per day across the window
# (partition="daily" implies this), or are there gaps?
q("""
    SELECT source_currency, target_currency,
           min(rate_date) AS first_date, max(rate_date) AS last_date,
           count(DISTINCT rate_date) AS days_with_a_rate,
           (max(rate_date) - min(rate_date) + 1) AS calendar_days_in_span
    FROM silver.dim_fx_rates
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

## 8. `rate_source` (`source` in Bronze) -- free text spot check

In [ ]:
q("SELECT rate_source, count(*) AS n FROM silver.dim_fx_rates GROUP BY 1 ORDER BY n DESC")

## 9. `silver.v_fx_latest_to_usd` -- the derived view other tables rely on for currency conversion

This view isn't a table `contracts.py` gates directly, but `products` (and possibly others) join against it for USD conversion when there's no natural as-of date. A bug here would silently propagate into every `amount_usd`-style estimate built from it.

In [ ]:
print("The view's contents:")
display(q("SELECT * FROM silver.v_fx_latest_to_usd ORDER BY source_currency"))

In [ ]:
# Does it actually pick the latest rate_date per source_currency, restricted to target_currency='USD'?
# Reproduce it independently (max date per currency, joined back) and diff against the view.
reproduced = q("""
    WITH latest AS (
        SELECT source_currency, max(rate_date) AS rate_date
        FROM silver.dim_fx_rates
        WHERE target_currency = 'USD'
        GROUP BY 1
    )
    SELECT f.source_currency, f.exchange_rate AS rate_to_usd, f.rate_date
    FROM silver.dim_fx_rates f
    JOIN latest l USING (source_currency, rate_date)
    WHERE f.target_currency = 'USD'
    ORDER BY f.source_currency
""")
view_contents = q("SELECT * FROM silver.v_fx_latest_to_usd ORDER BY source_currency")
print("Reproduced by hand matches the view exactly:" , reproduced.equals(view_contents))
display(reproduced)

In [ ]:
# Does every declared currency actually have a rate to USD at all? If MXN/COP/ARS -> USD is ever
# missing, v_fx_latest_to_usd silently omits that currency and any join against it for that
# currency returns nothing (not an error -- just a silent gap).
declared_currencies = {"MXN", "COP", "ARS", "USD"}
covered = set(q("SELECT DISTINCT source_currency AS v FROM silver.dim_fx_rates WHERE target_currency = 'USD'")["v"])
print("Currencies with NO rate to USD anywhere in the table:", declared_currencies - covered - {"USD"} or "none -- full coverage")

## Checklist

- [ ] Column inventory (1): rename mapping (`date`->`rate_date`, `source`->`rate_source`) correctly applied, types correct
- [ ] Row reconciliation (2) + dedup (2.1): 13,164 raw rows (1,097 dates x 12 pairs; the dictionary says 3,000), composite-key dedup keeps the most-recently-ingested row
- [ ] NOT NULL audit (3): `date`/`source_currency`/`target_currency`/`exchange_rate` fully populated, no cast-introduced nulls
- [ ] Nullable columns (4): `buy_rate`/`sell_rate`/`source` population rates sane, no cast silently dropped a value
- [ ] Currency domain (5): `source_currency`/`target_currency` only ever MXN/COP/ARS/USD, no same-to-same rows
- [ ] Rate sanity (6): no non-positive rates, `buy_rate <= exchange_rate <= sell_rate` holds where populated
- [ ] Date coverage (7): no window violations, no unexpected gaps in daily coverage per currency pair
- [ ] `rate_source` (8): values make sense
- [ ] `v_fx_latest_to_usd` (9): reproduces exactly by hand, every currency has a path to USD

In [ ]:
con.close()
print("closed")